# AnimTOON 3B — v3.1 / v4.1 Training on Kaggle

Both base model AND training data are pulled from HuggingFace at runtime.
No Kaggle dataset attachment needed — only the HF_TOKEN secret.

## One-time setup

1. **Add HF token as Kaggle secret**: Add-ons → Secrets → create `HF_TOKEN`
   (token must have read access to `srk0102200/AnimTOON-3B-v4` and
   `srk0102200/animtoon-iconscout-v4`)

2. **Enable GPU**: Settings → Accelerator → GPU T4 x2 (or GPU P100)

## Per run

1. Set `BASE_MODEL = "v4"` (or `"v3"`) in Cell 2
2. Run all — training takes ~2-3 hours
3. LoRA adapter saves to `/kaggle/working/animtoon-3b-vN-adapter/final/`

## Cell 1 — Install dependencies

In [ ]:
%%capture
# Unsloth gives 2x speedup + 70% less VRAM. Works on single T4 / P100.
!pip install -U unsloth trl==0.12.0 peft==0.13.2 accelerate bitsandbytes sentencepiece

## Cell 2 — Pick base model (v3 or v4)

In [ ]:
# === CHANGE THIS to switch between experiments ===
BASE_MODEL = "v4"  # "v3" or "v4"

MODEL_MAP = {
    "v3": "srk0102200/AnimTOON-3B",      # public, Qwen2.5-3B + v3 LoRA merged
    "v4": "srk0102200/AnimTOON-3B-v4",   # private, v4 integer format
}
MODEL_NAME = MODEL_MAP[BASE_MODEL]
OUTPUT_NAME = f"animtoon-3b-{BASE_MODEL}1-adapter"

# Training hyperparameters (same for both experiments for fair A/B)
MAX_SEQ_LEN = 2048      # prompt + output + chat template overhead
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EPOCHS = 2
BATCH_SIZE = 2
GRAD_ACCUM = 8          # effective batch = 16
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05

print(f"Base model : {MODEL_NAME}")
print(f"Output     : /kaggle/working/{OUTPUT_NAME}")

## Cell 3 — Load HF token + log in

In [ ]:
import os

# For API-pushed notebooks: paste HF_TOKEN in the cloud editor BEFORE running.
# DO NOT commit the real token to git. Leave blank in the committed version.
HF_TOKEN_OVERRIDE = ""  # <-- paste token here in the Kaggle editor

if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE

# Try Kaggle Secrets (works for UI-started kernels)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception:
    if os.environ.get("HF_TOKEN"):
        print("Using HF_TOKEN from HF_TOKEN_OVERRIDE")
    else:
        raise RuntimeError(
            "No HF_TOKEN available. Either paste into HF_TOKEN_OVERRIDE "
            "or pin HF_TOKEN via Add-ons > Secrets."
        )

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("Logged into HuggingFace")

## Cell 4 — Load base model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,             # auto: bf16 on Ampere+, fp16 on T4
    load_in_4bit=True,      # QLoRA — fits T4 easily
    token=os.environ.get("HF_TOKEN"),
)
print(f"Loaded: {MODEL_NAME}")
print(f"Tokenizer vocab: {len(tokenizer)}")

## Cell 5 — Attach LoRA adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
)
model.print_trainable_parameters()

## Cell 6 — Load + format training data

In [ ]:
import json
from pathlib import Path
from datasets import Dataset
from huggingface_hub import hf_hub_download

# Pull the training JSONL from HuggingFace (no Kaggle dataset needed)
DATA_REPO = "srk0102200/animtoon-iconscout-v4"
DATA_FILE = "iconscout_training_v4.jsonl"

print(f"Downloading {DATA_FILE} from {DATA_REPO} ...")
data_path = hf_hub_download(
    repo_id=DATA_REPO,
    filename=DATA_FILE,
    repo_type="dataset",
    token=os.environ.get("HF_TOKEN"),
)
print(f"Loaded from: {data_path}")

records = []
with open(data_path, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        records.append({"prompt": r["prompt"], "output": r["output"]})
print(f"Loaded {len(records)} training records")
print(f"Sample prompt: {records[0]['prompt'][:100]}")
print(f"Sample output (first 200 chars):\n{records[0]['output'][:200]}")

In [ ]:
# Format using chat template (same as v3 training)
def format_record(r):
    messages = [
        {"role": "user",      "content": f"Generate AnimTOON animation: {r['prompt']}"},
        {"role": "assistant", "content": r["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = Dataset.from_list(records)
dataset = dataset.map(format_record, remove_columns=dataset.column_names)

# 95/5 train/eval split
split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)}")
print(f"Eval:  {len(eval_dataset)}")

# Check token length distribution
sample = train_dataset.select(range(min(500, len(train_dataset))))
lengths = [len(tokenizer(x["text"])["input_ids"]) for x in sample]
print(f"Token length on 500 samples — min={min(lengths)} max={max(lengths)} avg={sum(lengths)/len(lengths):.0f}")

## Cell 7 — Set up Trainer

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    args=SFTConfig(
        output_dir=f"/kaggle/working/{OUTPUT_NAME}",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        max_grad_norm=1.0,
        optim="adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=2,
        eval_strategy="steps",
        eval_steps=500,
        load_best_model_at_end=True,
        report_to="none",
        dataset_text_field="text",
        seed=42,
    ),
)
print("Trainer ready.")

## Cell 8 — Train (~2-3 hours)

In [ ]:
trainer_stats = trainer.train()
print(f"Training complete. Final loss: {trainer_stats.training_loss:.4f}")

## Cell 9 — Save adapter

In [ ]:
save_path = f"/kaggle/working/{OUTPUT_NAME}/final"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Saved LoRA adapter + tokenizer to: {save_path}")
!ls -lh /kaggle/working/{OUTPUT_NAME}/final/

## Cell 10 — Quick inference sanity check

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "a red circle pulsing in the center",
    "a businessman waving hello with his right arm",
    "14-layer cat with head, body, 4 legs, tail. tail wags, ears twitch",
    "a person walking forward with swinging arms and stepping legs",
]

for p in test_prompts:
    messages = [{"role": "user", "content": f"Generate AnimTOON animation: {p}"}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=1024, temperature=0.5,
            top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    result = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n=== {p} ===")
    print(result[:500])

## Cell 11 — Zip adapter for download

After this cell runs, use the Kaggle sidebar **Output** panel on the right to download the zip file.

In [ ]:
import shutil
zip_path = f"/kaggle/working/{OUTPUT_NAME}.zip"
shutil.make_archive(
    base_name=f"/kaggle/working/{OUTPUT_NAME}",
    format="zip",
    root_dir=f"/kaggle/working/{OUTPUT_NAME}/final",
)
import os
size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"Adapter zipped: {zip_path} ({size_mb:.1f} MB)")
print("Download it from the Output panel on the right sidebar.")